# df_inventory - Assisted Housing Property Details: Cleaning & Exploration

`df_inventory` is the core property-level dataset in the AHI. Each row represents one **assisted housing development** in Alachua County - a residential property receiving some form of government or quasi-government support. With 168 columns, it covers everything from physical property characteristics and funding program flags to tenant demographics and neighborhood context drawn from the Census.

Because funding is tracked at the property level, a single development can be supported by multiple programs simultaneously. This makes the dataset richer but also more complex: a property may appear fully funded by one measure while carrying expiring contracts in another.

#### Column Groups

| Group | Key Columns | Description |
|-------|-------------|-------------|
| **Identifiers** | `Shim ID`, `FHFC Key`, `HUD REMS`, `Florida DOR Parcel` | IDs used by different agencies to track the same property |
| **Location** | `Development Name`, `Address`, `City`, `Zip Code`, `Census Tract`, `Latitude`/`Longitude` | Where the property is and how it maps to Census geographies |
| **Unit Counts** | `Total Units`, `Assisted Units`, `HUD/RD Rental Assistance Units`, `Public Housing ACC Units`, bedroom breakdowns | How many units exist and how many are actively assisted, broken down by size and assistance type |
| **Funder Flags** | `FHFC Funded`, `HUD Multifamily Funded`, `HUD Public Housing Funded`, `RD Funded`, `LHFA Funded` | Boolean indicators for which funders are attached to the property |
| **Programs** | `Housing Programs`, `Has Housing Credits 4%/9%`, `Has SAIL`, `Has Section 515`, etc. | Individual program flags and freetext program list; a property may carry multiple programs |
| **Program Timing** | Funding years, contract start dates, expiration dates per program | When each program began and when assistance is set to expire |
| **Property Details** | `Year Built`, `Owner Type`, `Construction Type`, `Total Living Area`, `Number of Buildings`, `REAC Score` | Physical and ownership characteristics; REAC scores apply only to HUD-funded properties |
| **Financial / Rent** | AMI-tiered units, average rent per unit by bedroom size, `Average Rent/FMR Ratio`, utility allowances, sale price, just value | Rent levels relative to Fair Market Rent and distribution of units across income tiers |
| **Affordability Timeline** | `Affordability Start Date`, `Overall year of subsidy expiration`, `FHFC Preservation Set-Aside`, `In QCT 2022`, `In DDA 2022`, `RAD conversion` | When affordability began, when it may end, and flags for preservation risk |
| **Tenant Demographics** | Household size, % elderly, % with children, race/ethnicity, income distribution, AMI brackets | Characteristics of households currently living in the property |
| **Neighborhood Context** | Vacancy rates, homeownership/renter rates, racial demographics, housing values - at tract and county level | Census-derived neighborhood context, paired with county-level benchmarks for comparison |

**Area Median Income (AMI)** is the midpoint household income for a given region, used as a benchmark for affordability. Units at or below 30% AMI serve the lowest-income households; units above 80% AMI are generally considered market-rate even if they fall within an assisted property.

**Fair Market Rent (FMR)** is a HUD-published estimate of what a modest rental unit should cost in a given market. The `Average Rent/FMR ratio` compares a property's rents to this benchmark - a value below 1.0 suggests rents are being held below market rate, likely through a subsidy.

**REAC** (Real Estate Assessment Center) is HUD's inspection program. REAC scores rate the physical condition of HUD-assisted properties on a 0-100 scale.

**QCT** (Qualified Census Tract) and **DDA** (Difficult Development Area) are HUD-designated geographic categories used to allocate additional tax credit equity to affordable housing projects in high-cost or low-income areas.

**RAD** (Rental Assistance Demonstration) is a HUD program that converts older public housing contracts to long-term project-based rental assistance, often as part of a renovation or ownership transition.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.ticker import StrMethodFormatter

df_inventory = pd.read_csv("../Data/Cleaned Data/df_inventory.csv")
df_inventory.head()

## Funding Coverage

How is funding distributed across the 60 developments, and is every development fully covered by its assistance programs?

### Properties per Funder and Multi-Funder Overlap

`df_funder` showed us how many properties and units each funder covers in aggregate. Here we continue that analysis at the development level using `df_inventory`, which has a row for each individual development. We start with two simple bar charts to understand how funding is spread across the 60 developments in the dataset.

The first chart counts how many developments each funder flag covers - a development-level view of each funder's reach in Alachua County. The second shows how many funder flags each development carries, which tells us how often funding from multiple sources is stacked on the same development.

In [ ]:
funder_cols = ["FHFC Funded", "HUD Multifamily Funded", "RD Funded", "LHFA Funded", "HUD Public Housing Funded"]

funder_labels = {
    "A": "Florida Housing Finance Corporation",
    "B": "HUD Multifamily",
    "C": "USDA Rural Development",
    "D": "Local Housing Finance Authority",
    "E": "HUD Public Housing"
}
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
codes = list(funder_labels.keys())
full_names = list(funder_labels.values())

# Chart 1: Properties per funder
prop_counts = [df_inventory[col].sum() for col in funder_cols]

legend_handles = [Patch(color=colors[i], label=f"{code}: {full_names[i]}") for i, code in enumerate(codes)]

fig, ax = plt.subplots(figsize=(7, 6))
bars = ax.bar(codes, prop_counts, color=colors)
ax.bar_label(bars, fmt="%g", fontsize=11, padding=3)
ax.set_title("df_inventory - Properties per Funder")
ax.set_ylabel("Number of Properties")
ax.legend(handles=legend_handles, loc="upper right", fontsize=10, frameon=True, handlelength=2, handleheight=1.5, labelspacing=0.8)
plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_properties_per_funder.png", dpi=150, bbox_inches="tight")
plt.show()

# Chart 2: Number of funders per property
funder_count_per_property = df_inventory[funder_cols].sum(axis=1)
distribution = funder_count_per_property.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(distribution.index.astype(str), distribution.values, color="#4C72B0")
for i, v in enumerate(distribution.values):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=11)
ax.set_title("df_inventory - Number of Funders per Property")
ax.set_xlabel("Number of Funders")
ax.set_ylabel("Number of Properties")
plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_funder_count_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observations

**Properties per Funder:**
- FHFC funds the most developments at 36, nearly matching all other funders combined.
- HUD Multifamily is a distant second at 23, with LHFA (6), USDA Rural Development (6), and HUD Public Housing (5) each covering a small share of the portfolio.
- These counts match `df_funder` exactly, confirming the two datasets are consistent at the development level.

**Number of Funders per Development:**
- Most developments (40 out of 60, 67%) are funded by exactly one source.
- 16 developments (27%) have two funders and one has three, showing that layered funding does exist but is not the norm.
- 3 developments show no funder flags at all, which is unexpected and likely reflects a data gap.

> **NOTE:** Investigate the 3 developments with no funder flags - they should have at least one funding source. Also worth examining which funders most commonly appear together on the same development.

### Which Developments Account for the Unassisted Unit Gap?

Across all 60 developments, there are 393 units that fall within a funded development but are not receiving assistance. Rather than being spread evenly, this gap is likely driven by a small number of developments. To test this, we use a **Pareto chart** - a bar chart sorted from largest to smallest, with a line showing the running cumulative percentage on a second axis. A dashed line marks the 80% threshold, which is the classic benchmark for identifying the "vital few" causes behind a problem.

In [ ]:
unassisted = (df_inventory["Total Units"] - df_inventory["Assisted Units"]).dropna()
unassisted = unassisted[unassisted > 0].sort_values(ascending=False)
names = df_inventory.loc[unassisted.index, "Development Name"].values
n = len(unassisted)

codes = [str(i + 1) for i in range(n)]
cmap = plt.cm.get_cmap("tab20", n)
colors = [cmap(i) for i in range(n)]

cumulative_pct = unassisted.cumsum() / unassisted.sum() * 100

fig, ax1 = plt.subplots(figsize=(10, 5))

bars = ax1.bar(codes, unassisted.values, color=colors)
ax1.bar_label(bars, fmt="%g", fontsize=9, padding=3)
ax1.set_ylabel("Unassisted Units")
ax1.set_title("df_inventory - Pareto Chart of Unassisted Units per Property")

ax2 = ax1.twinx()
ax2.plot(codes, cumulative_pct.values, color="black", marker="o", markersize=4, linewidth=1.5)
ax2.axhline(80, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("Cumulative %")
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}%"))

legend_handles = [Patch(color=colors[i], label=f"{codes[i]}: {names[i]}") for i in range(n)]
ax1.legend(handles=legend_handles, loc="center right", fontsize=9, frameon=True, handlelength=2, handleheight=1.5, labelspacing=0.8)

plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_unassisted_pareto.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observations

- **Reserve at Kanapaha** alone accounts for 216 unassisted units - 55% of the total gap by itself.
- Adding **Woodland Park / Eastwood Meadows** (92 units) brings the cumulative to 78.4%, just under the 80% threshold.
- **Brookside** (52 units) pushes it to 91.6%, well past the threshold.
- The remaining 5 developments together account for only 33 units (8.4% combined).

The gap between total and assisted units in Alachua County is not a widespread issue - it is almost entirely driven by 2-3 developments. Whether this reflects mixed-income development design, expired assistance contracts, or another cause would require looking at those developments individually.

## Who Is Served

What kinds of households does the assisted housing stock serve - by declared population type, income level, and unit size?

### Who Does Assisted Housing Serve? Units by Target Population

Each development in `df_inventory` is tagged with a `Target Population` describing who it is designed to serve. The possible values are **Family**, **Elderly**, **Persons with Disabilities**, and **Link** - where Link refers to units funded by FHFC and set aside for extremely low income households with special needs. A semicolon between values means a development serves multiple populations (e.g. `Elderly;Family` accepts both elderly residents and families).

Because units within a multi-population development are not broken down by population in this dataset, we treat each category as a whole. We use a **Pareto chart** to show which target populations account for the most units and how quickly they accumulate toward 100%.

In [ ]:
target_units = df_inventory.groupby("Target Population")["Total Units"].sum().sort_values(ascending=False)
labels = list(target_units.index)
values = list(target_units.values)
n = len(labels)

codes = [str(i + 1) for i in range(n)]
cmap = plt.cm.get_cmap("tab20", n)
colors = [cmap(i) for i in range(n)]

cumulative_pct = target_units.cumsum() / target_units.sum() * 100

fig, ax1 = plt.subplots(figsize=(10, 5))

bars = ax1.bar(codes, values, color=colors)
ax1.bar_label(bars, fmt="%g", fontsize=10, padding=3)
ax1.set_ylabel("Total Units")
ax1.set_title("df_inventory - Pareto Chart of Total Units by Target Population")

ax2 = ax1.twinx()
ax2.plot(codes, cumulative_pct.values, color="black", marker="o", markersize=4, linewidth=1.5)
ax2.axhline(80, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("Cumulative %")
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}%"))

legend_handles = [Patch(color=colors[i], label=f"{codes[i]}: {labels[i]}") for i in range(n)]
ax1.legend(handles=legend_handles, loc="center right", fontsize=9, frameon=True, handlelength=2, handleheight=1.5, labelspacing=0.8)

plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_target_population_pareto.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observations

- **Family** alone accounts for 2,271 units - 48% of all 4,731 units in the dataset.
- Adding **Elderly;Family** (841 units) brings the cumulative to 65.8%, and **Family;Link** (764 units) pushes it to 81.9%, just crossing the 80% threshold.
- Unlike the unassisted units Pareto - where one development dominated at 55% - the distribution here is more gradual. Each of the top three categories contributes meaningfully rather than one dwarfing the rest.
- Every one of the top five categories includes "Family," confirming that family-oriented housing is the dominant model in Alachua County's assisted housing stock.
- Pure **Elderly** (121 units) and **Persons with Disabilities** (125 units) are each small in isolation, though Elderly appears as a component in two other categories (bars 2 and 4).

### Unit Distribution by AMI Tier

The AMI tier columns (`Units <=35% AMI` through `Units >80% AMI`) break down how many units in each development are reserved for households at different income levels. Summing across all developments gives a picture of which income groups the assisted housing stock is actually built to serve.

**Area Median Income (AMI)** tiers are used to set maximum income limits for assisted housing. A unit at 60% AMI, for example, can only be rented to a household earning at or below 60% of the area median income.

In [ ]:
ami_cols = ["Units <=35% AMI", "Units 40-50% AMI", "Units 55-60% AMI", "Units 65-80% AMI", "Units >80% AMI"]
ami_labels = ["≤35%", "40-50%", "55-60%", "65-80%", ">80%"]
ami_totals = [df_inventory[col].sum() for col in ami_cols]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(ami_labels, ami_totals, color="#4C72B0")
ax.bar_label(bars, fmt="%g", fontsize=10, padding=3)
ax.set_title("df_inventory - Total Units by AMI Tier")
ax.set_xlabel("AMI Tier")
ax.set_ylabel("Total Units")
plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_ami_tier_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Redraw AMI tier chart with bar widths proportional to interval size.
# The <=35% bin spans 35 percentage points; 40-50% spans 10; 55-60% spans 5;
# 65-80% spans 15; >80% is open-ended, treated as 20 (up to 100%).
# Gaps between tiers (35-40, 50-55, 60-65) are shown as empty space on the x-axis.

ami_intervals = [
    ("≤35%",  0,  35),
    ("40-50%", 40, 50),
    ("55-60%", 55, 60),
    ("65-80%", 65, 80),
    (">80%",   80, 100),
]

fig, ax = plt.subplots(figsize=(9, 5))

for (label, lo, hi), count in zip(ami_intervals, ami_totals):
    width = hi - lo
    ax.bar(lo, count, width=width, align="edge", color="#4C72B0", edgecolor="white")
    ax.text(lo + width / 2, count + 30, f"{count:g}", ha="center", va="bottom", fontsize=10)

ax.set_title("df_inventory - Total Units by AMI Tier (Width Proportional to Interval)")
ax.set_xlabel("AMI (%)")
ax.set_ylabel("Total Units")
ax.set_xlim(0, 100)
ax.set_xticks([0, 35, 40, 50, 55, 60, 65, 80, 100])
ax.set_xticklabels(["0%", "35%", "40%", "50%", "55%", "60%", "65%", "80%", "100%"], fontsize=8)

plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_ami_tier_distribution_proportional.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observations

- The 55-60% AMI tier dominates with 2,445 units - far exceeding all other tiers combined (691 units).
- The ≤35% (291) and 40-50% (333) tiers are relatively thin, and the 65-80% tier has only 67 units. There are no units reported above 80% AMI.
- The spike at 55-60% AMI almost certainly reflects FHFC's Housing Credit programs, which use 60% AMI as their standard income limit. Most FHFC-financed developments are built specifically to that threshold, making it the dominant tier by design.
- The implication is that assisted housing in Alachua County is primarily serving moderate-low income households rather than the very lowest income households (≤35% AMI), where need is arguably greatest.

### Unit Size Distribution

The bedroom breakdown columns (`Number of 0 BR` through `Number of 4 or more BR`) tell us how many units of each size exist across all developments. Summing these across the dataset gives a picture of what household sizes the assisted housing stock is built to serve.

In [ ]:
br_cols = ["Number of 0 BR", "Number of 1 BR", "Number of 2 BR", "Number of 3 BR", "Number of 4 or more BR"]
br_labels = ["0 BR", "1 BR", "2 BR", "3 BR", "4+ BR"]
br_totals = [df_inventory[col].sum() for col in br_cols]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(br_labels, br_totals, color="#4C72B0")
ax.bar_label(bars, fmt="%g", fontsize=10, padding=3)
ax.set_title("df_inventory - Total Units by Bedroom Size")
ax.set_ylabel("Total Units")
plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_bedroom_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observations

- The distribution is bell-shaped, peaking at **2 BR** (1,794 units), with 1 BR (1,224) and 3 BR (1,403) flanking it on either side.
- Studios (0 BR, 85 units) and 4+ BR units (74 units) are rare, indicating the stock is not strongly oriented toward single individuals or very large households.
- This shape is consistent with the target population findings - Family being the dominant category naturally produces a distribution centered on 2- and 3-bedroom units suited for small families.

## Housing Economics

What does it cost to live in assisted housing, and do property valuations hold up under scrutiny?

### Rent Distribution by Bedroom Size

To understand how rents vary across developments, we use a **box and whisker plot** - one box per bedroom size. Each data point is a development's average rent per unit for that bedroom size, so the box shows how consistently rents are set across developments rather than variation within a single development. Developments with no units of a given size are excluded.

A **box and whisker plot** shows five summary values: the minimum, the 25th percentile (bottom of the box), the median (line inside the box), the 75th percentile (top of the box), and the maximum. Points beyond the whiskers are outliers.

In [ ]:
rent_cols = ["0 BR Av. Rent ($)", "1 BR Av. Rent ($)", "2 BR Av. Rent ($)", "3 BR Av. Rent ($)", "4+ BR Av. Rent ($)"]
rent_labels = ["0 BR", "1 BR", "2 BR", "3 BR", "4+ BR"]
rent_data = [df_inventory[col].dropna() for col in rent_cols]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(rent_data, labels=rent_labels, patch_artist=True,
           boxprops=dict(facecolor="#4C72B0", alpha=0.6),
           medianprops=dict(color="black", linewidth=2))
ax.set_title("df_inventory - Rent Distribution by Bedroom Size")
ax.set_xlabel("Bedroom Size")
ax.set_ylabel("Average Rent ($)")

fig.text(
    0.5, -0.04,
    "Note: 0 BR is based on exactly 2 data points and should not be interpreted as representative.",
    ha="center", fontsize=8, style="italic", color="gray"
)

plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_rent_by_bedroom.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observations

- Rents increase consistently from 1 BR through 4+ BR, as expected - larger units cost more.
- The spread (box height) also tends to widen with bedroom size, suggesting more variation in how developments price larger units.
- **0 BR** appears elevated but is based on exactly 2 data points and is not reliable enough to draw conclusions from.

> **NOTE:** Re-examine 0 BR rents if more data becomes available. With only 2 developments reporting studio rents, the box is not representative.

### Rent as a Percentage of Fair Market Rent

The `Average Rent/FMR ratio` column expresses each development's average rent as a percentage of HUD's Fair Market Rent for Alachua County. A value below 100 means rents are held below what a comparable unit costs on the open market - the defining feature of a subsidized development. A value above 100 means rents exceed the local market benchmark, which is unusual for assisted housing and warrants scrutiny.

22 of the 58 developments with data report a ratio of 0. These are almost certainly public housing and similar developments where rent is calculated as a share of household income rather than set against a fixed market benchmark - making the FMR ratio meaningless for them. They are excluded from the chart below and noted separately.

In [ ]:
ratio_data = pd.to_numeric(df_inventory["Average Rent/FMR ratio"], errors="coerce").dropna()
nonzero = ratio_data[ratio_data > 0].sort_values()

below_fmr = (nonzero < 100).sum()
at_or_above = (nonzero >= 100).sum()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(nonzero, bins=15, color="#4C72B0", edgecolor="white")
ax.axvline(100, color="#C44E52", linestyle="--", linewidth=1.5, label="FMR (100%)")
ax.set_title("df_inventory - Average Rent as % of Fair Market Rent (zeros excluded)")
ax.set_xlabel("Average Rent / Fair Market Rent (%)")
ax.set_ylabel("Number of Developments")
ax.legend()

fig.text(
    0.5, -0.04,
    "Note: 22 of the 58 developments report a ratio of 0 and are excluded. This may reflect a data entry error or developments where tenants did not report income.",
    ha="center", fontsize=8, style="italic", color="gray"
)

plt.tight_layout()
plt.savefig("../Outputs/df_inventory/df_inventory_rent_fmr_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Developments excluded (ratio = 0): {(ratio_data == 0).sum()}")
print(f"Below FMR (<100%):     {below_fmr} developments  |  range: {nonzero[nonzero < 100].min():.0f}%-{nonzero[nonzero < 100].max():.0f}%")
print(f"At or above FMR (≥100%): {at_or_above} developments  |  range: {nonzero[nonzero >= 100].min():.0f}%-{nonzero[nonzero >= 100].max():.0f}%")

#### Observations

- Of the 36 developments with a non-zero FMR ratio, **24 (67%) charge below market rate** - ranging from 51% to 94% of FMR - confirming meaningful rent subsidization across most of the stock.
- **12 developments (33%) charge at or above FMR**, with rents ranging from 102% to 135% of the local market benchmark. This is unexpected for assisted housing and likely reflects programs where the assistance is in the form of construction financing (such as FHFC Housing Credits) rather than an ongoing rent subsidy. The financing helps build the development but does not cap what tenants pay.
- The 22 zero-ratio developments are excluded but represent a meaningful portion of the stock. These are almost certainly income-based developments - primarily public housing - where rent is set as a percentage of household income and has no direct relationship to FMR.
- Together, the picture is mixed: roughly two-thirds of FMR-tracked developments are genuinely below market, but a substantial minority are not, and a large group cannot be evaluated on this metric at all.

In [ ]:
income_cols = [
    "% $0-4,999",
    "% $5,000-9,999",
    "% $10,000-14,999",
    "% $15,000-19,999",
    "% $20,000 and above"
]
income_labels = ["$0-4,999", "$5,000-9,999", "$10,000-14,999", "$15,000-19,999", "$20,000+"]

weights = df_inventory["Total Units"].fillna(0)
weighted_pcts = [
    (pd.to_numeric(df_inventory[col], errors="coerce").fillna(0) * weights).sum() / weights.sum()
    for col in income_cols
]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(income_labels, weighted_pcts, color="#4C72B0")
ax.bar_label(bars, fmt="%.1f%%", fontsize=10, padding=3)
ax.set_title("df_inventory - Household Income Distribution (Unit-Weighted)")
ax.set_xlabel("Annual Household Income")
ax.set_ylabel("% of Households (weighted by units)")
ax.tick_params(axis="x", labelrotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# FHFC 2025 income limits for Alachua County, 4-person household
ami_lines = {
    "35% ($36,400)":  36400,
    "50% ($52,000)":  52000,
    "60% ($62,400)":  62400,
    "80% ($83,200)":  83200,
}
ami_colors = ["#DD8452", "#55A868", "#C44E52", "#8172B2"]

avg_income = pd.to_numeric(df_inventory["Average Annual Household Income ($)"], errors="coerce")
weights = df_inventory["Total Units"].fillna(0)
valid = avg_income.notna() & (weights > 0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(avg_income[valid], bins=20, weights=weights[valid], color="#4C72B0", edgecolor="white")

for (label, val), color in zip(ami_lines.items(), ami_colors):
    ax.axvline(val, color=color, linestyle="--", linewidth=1.5, label=label)

ax.set_title("df_inventory - Average Household Income per Development (Unit-Weighted)\nFHFC 2025 AMI Tier Limits shown, 4-Person Household")
ax.set_xlabel("Average Annual Household Income ($)")
ax.set_ylabel("Total Units")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(title="AMI Tier Limit", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
pct_ami = pd.to_numeric(df_inventory["Average Annual Household Income (% AMI)"], errors="coerce").dropna()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pct_ami, bins=20, color="#4C72B0", edgecolor="white")
ax.set_title("df_inventory - Average Household Income (% AMI) Distribution")
ax.set_xlabel("Average Annual Household Income (% AMI)")
ax.set_ylabel("Number of Developments")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
plt.show()

In [ ]:
valid_mask = pd.to_numeric(df_inventory["Average Annual Household Income (% AMI)"], errors="coerce").notna()
pct_ami_vals = pd.to_numeric(df_inventory.loc[valid_mask, "Average Annual Household Income (% AMI)"], errors="coerce")
unit_weights = df_inventory.loc[valid_mask, "Total Units"].fillna(0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pct_ami_vals, bins=20, weights=unit_weights, color="#4C72B0", edgecolor="white")
ax.set_title("df_inventory - Average Household Income (% AMI) Distribution (Unit-Weighted)")
ax.set_xlabel("Average Annual Household Income (% AMI)")
ax.set_ylabel("Total Units")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
plt.show()